# D1S Decay-Photon Transport - ENDF/B-VIII.1

This notebook compares YAMC and OpenMC D1S (Direct 1-Step) decay-photon transport results using a broomstick (cylinder) geometry with the ENDF/B-VIII.1 nuclear data library.

## Model Description

- **Geometry**: Cylinder with radius 1 cm, height 100 cm (z = -50 to +50), vacuum boundaries
- **Source**: Point source at origin, 14.06 MeV monoenergetic neutrons, isotropic
- **Material**: Single nuclide at 1.0 g/cm³, temperature 294 K
- **Transport**: Coupled neutron-photon with decay photons (`use_decay_photons=True`)
- **Statistics**: 500,000 particles × 10 batches = 5,000,000 total histories
- **Timesteps**: 3 × 1 hour irradiation + 2 × 15 day cooling
- **Tallies**: TCF-corrected photon flux spectra (VITAMIN-J-42) at each timestep

In [1]:
# Setup and imports
import re
import time
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
import glob
import os
import yamc
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown

# Configuration
NUCLEAR_DATA_DIR = {
    "endf-b8.1": "/home/jon/nuclear_data/endf-b8.1-hdf5/endfb-viii.1-hdf5/",
}

NUCLEAR_DATA = "endf-b8.1"
PARTICLES = 5000000
BATCHES = 10
SOURCE_ENERGY = 14.06e6  # 14.06 MeV neutrons
FLUX_DECADES = 6

# Optional smoke-test cap on the iteration list (env var; unset/0 = full).
NUMBER_OF_SIMULATIONS = int(os.environ.get("YAMC_VERIFY_NUMBER_OF_SIMULATIONS", "0")) or None

# Codes/backends to run. Comma-separated list via the MC_BACKENDS env var.
# Tokens:
#   openmc-cpu  -- reference OpenMC simulation
#   yamc-cpu    -- YAMC D1S on CPU
#   yamc-gpu    -- YAMC D1S on GPU (requires a working f64 Vulkan adapter)
# Default runs all three. The GPU dispatch raises a clean RuntimeError if no
# f64 Vulkan adapter is present, which is enough signal for CI / interactive use.
_VALID_BACKENDS = ("openmc-cpu", "yamc-cpu", "yamc-gpu")
_raw = os.environ.get("MC_BACKENDS", "openmc-cpu,yamc-cpu,yamc-gpu")
MC_BACKENDS = [b.strip().lower() for b in _raw.split(",") if b.strip()]
_bad = [b for b in MC_BACKENDS if b not in _VALID_BACKENDS]
if _bad:
    raise ValueError(
        f"MC_BACKENDS contains unknown token(s) {_bad}; "
        f"valid tokens are {list(_VALID_BACKENDS)}"
    )
if not MC_BACKENDS:
    raise ValueError("MC_BACKENDS resolved to an empty list")
RUN_OPENMC = "openmc-cpu" in MC_BACKENDS
COMPUTES = [b.split("-", 1)[1] for b in MC_BACKENDS if b.startswith("yamc-")]
print(f"Backends: {MC_BACKENDS}")

# Cylinder dimensions
CYLINDER_RADIUS = 1.0  # cm
CYLINDER_HALF_HEIGHT = 50.0  # cm

# Group structure for photon spectra
PHOTON_GROUP_STRUCTURE = "VITAMIN-J-42"

# Transmutation data: OpenMC reads the XML chain; YAMC reads the cached arrow
# directory (auto-downloaded into ~/.cache/yamc/transmutation-endf-b8.1-sfr.arrow/).
# OpenMC reads the XML depletion chain. The primary path matches the other
# tasks; if it is absent, fall back to any installed openmc_data copy of the same
# b8.1 chain so the OpenMC D1S run still works on this machine.
_CHAIN_CANDIDATES = [
    "/home/jon/yamc-org/openmc_data/src/openmc_data/depletion/chain_endf_b8.1.xml",
]
try:
    import openmc_data as _omc_data
    _CHAIN_CANDIDATES.append(
        os.path.join(os.path.dirname(_omc_data.__file__), "depletion", "chain_endf_b8.1.xml")
    )
except Exception:
    pass
_CHAIN_CANDIDATES += sorted(glob.glob(
    "/home/jon/*/.venv/lib/python*/site-packages/openmc_data/depletion/chain_endf_b8.1.xml"
))
CHAIN_FILE_OPENMC = next((p for p in _CHAIN_CANDIDATES if os.path.exists(p)), _CHAIN_CANDIDATES[0])
TRANSMUTATION_DATA = os.path.expanduser("~/.cache/yamc/transmutation-endf-b8.1-sfr.arrow")
TRANSMUTE_REDUCE_LEVEL = 4

# D1S parameters
D1S_TIMESTEPS = [3600.0, 3600.0, 3600.0, 15 * 86400.0, 15 * 86400.0]
D1S_SOURCE_RATES = [1.0, 1.0, 1.0, 0.0, 0.0]
D1S_PHASES = ["irrad", "irrad", "irrad", "cool", "cool"]

Backends: ['openmc-cpu', 'yamc-cpu', 'yamc-gpu']


In [2]:
# OpenMC results cache - load reference results from disk if a previous
# run produced them with the same parameters; otherwise run OpenMC and
# write the results out for next time.
import sys
from pathlib import Path as _Path

_repo_root = _Path.cwd().resolve()
while _repo_root != _repo_root.parent and not (_repo_root / "cache_utils.py").exists():
    _repo_root = _repo_root.parent
if (_repo_root / "cache_utils.py").exists() and str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))
from cache_utils import cached_or_run

CACHE_DIR = _Path("cache")
OPENMC_CACHE_META = {
    "library": NUCLEAR_DATA,
    "particles": PARTICLES,
    "batches": BATCHES,
    "source_energy": SOURCE_ENERGY,
    "photon_group_structure": PHOTON_GROUP_STRUCTURE,
    "cylinder_radius": CYLINDER_RADIUS,
    "cylinder_half_height": CYLINDER_HALF_HEIGHT,
    "chain_file": CHAIN_FILE_OPENMC,
    "d1s_timesteps": list(D1S_TIMESTEPS),
    "d1s_source_rates": list(D1S_SOURCE_RATES),
}

In [3]:
def run_openmc_d1s(nuclide, neutron_path, element, photon_path):
    """Run OpenMC D1S simulation with broomstick geometry.

    Returns dict with TCF-corrected photon flux spectra at each timestep.
    """
    import openmc
    from openmc.deplete import d1s

    openmc.reset_auto_ids()
    openmc.config['cross_sections'] = os.path.join(NUCLEAR_DATA_DIR[NUCLEAR_DATA], 'cross_sections.xml')
    # OpenMC keeps the full chain - its fission-product-yield borrowing needs
    # nuclides outside any small reduce(...) reach.
    openmc.config['chain_file'] = CHAIN_FILE_OPENMC

    material = openmc.Material()
    material.temperature = 294
    material.add_nuclide(nuclide, 1.0)
    material.set_density('g/cm3', 1.0)

    cylinder = openmc.ZCylinder(r=CYLINDER_RADIUS, boundary_type='vacuum')
    z_bot = openmc.ZPlane(z0=-CYLINDER_HALF_HEIGHT, boundary_type='vacuum')
    z_top = openmc.ZPlane(z0=CYLINDER_HALF_HEIGHT, boundary_type='vacuum')
    region = -cylinder & +z_bot & -z_top

    cell = openmc.Cell(name='broomstick', region=region, fill=material)
    geometry = openmc.Geometry([cell])

    source = openmc.IndependentSource(
        space=openmc.stats.Point((0, 0, 0)),
        angle=openmc.stats.Isotropic(),
        energy=openmc.stats.Discrete([SOURCE_ENERGY], [1]),
    )

    settings = openmc.Settings(
        particles=PARTICLES,
        batches=BATCHES,
        source=source,
        seed=1,
        run_mode='fixed source',
        output={'tallies': False, 'summary': False},
    )
    settings.photon_transport = True
    settings.use_decay_photons = True

    cell_filter = openmc.CellFilter(cell)
    photon_filter = openmc.ParticleFilter(['photon'])
    energy_filter = openmc.EnergyFilter.from_group_structure(PHOTON_GROUP_STRUCTURE)
    energy_bins_2d = energy_filter.bins
    energy_bins = np.concatenate([[energy_bins_2d[0, 0]], energy_bins_2d[:, 1]])

    tally = openmc.Tally(name='d1s_spectrum')
    tally.filters = [cell_filter, photon_filter, energy_filter]
    tally.scores = ['flux']

    model = openmc.Model(
        geometry, openmc.Materials([material]), settings,
        openmc.Tallies([tally]),
    )

    d1s.prepare_tallies(model=model)
    radionuclides = d1s.get_radionuclides(model)

    t0 = time.time()
    sp_path = model.run()
    elapsed = time.time() - t0

    with openmc.StatePoint(sp_path) as sp:
        tally_result = sp.get_tally(name='d1s_spectrum')

    tcf = d1s.time_correction_factors(
        nuclides=radionuclides, timesteps=D1S_TIMESTEPS,
        source_rates=D1S_SOURCE_RATES, timestep_units='s',
    )

    spectra = []
    spectra_std = []
    for step_idx in range(len(D1S_TIMESTEPS)):
        corrected = d1s.apply_time_correction(
            tally=tally_result, time_correction_factors=tcf,
            index=step_idx + 1, sum_nuclides=True,
        )
        spectra.append(corrected.mean.flatten().tolist())
        spectra_std.append(corrected.std_dev.flatten().tolist())

    return {
        'radionuclides': radionuclides,
        'energy_bins': energy_bins,
        'spectra': spectra,
        'spectra_std': spectra_std,
        'elapsed': elapsed,
    }

In [4]:
def run_yamc_d1s(nuclide, neutron_path, element, photon_path, compute="cpu"):
    """Run YAMC D1S simulation with broomstick geometry. Returns (results_dict, model)."""
    import yamc as mc

    material = mc.Material(
        composition={nuclide: 1.0},
        density=1.0,
        temperature=294,
        id=1,
    )
    material.read_nuclear_data('endf-b8.1')

    cylinder = mc.Cylinder(axis='z', radius=CYLINDER_RADIUS, boundary='vacuum')
    z_bot = mc.Plane(axis='z', offset=-CYLINDER_HALF_HEIGHT, boundary='vacuum')
    z_top = mc.Plane(axis='z', offset=CYLINDER_HALF_HEIGHT, boundary='vacuum')
    region = cylinder.below & z_bot.above & z_top.below

    cell = mc.Cell(id=1, name='broomstick', region=region, material=material)
    geometry = mc.Geometry([cell])

    source = mc.NeutronSource(
        position=(0, 0, 0),
        direction=mc.sources.Isotropic(),
        energy=mc.sources.Discrete([SOURCE_ENERGY], [1.0]),
    )

    # Reduce the YAMC transmutation network to nuclides reachable from the seed
    # (matches OpenMC's chain.reduce(...)); export to a local arrow dir and point
    # the global at it so radionuclides()/decay-photon channels pick it up.
    yamc_chain = mc.TransmutationChain(TRANSMUTATION_DATA)
    yamc_reduced = yamc_chain.reduce([nuclide], level=TRANSMUTE_REDUCE_LEVEL)
    reduced_path = os.path.join(os.getcwd(), '_reduced_transmutation.chain.arrow')
    yamc_reduced.export_to_arrow(reduced_path)
    mc.transmutation_data = reduced_path

    # Unstable activation products reachable from this model's materials.
    radionuclides = mc.Model(
        geometry=geometry, source=source, total_particles=1,
    ).radionuclides()

    tally = mc.Tally(
        scores=['flux'],
        name='d1s_spectrum',
        cells=cell,
        particle='photon',
        energy_group_structure=PHOTON_GROUP_STRUCTURE,
        parent_nuclides=radionuclides,
    )

    energy_bins = np.array(tally.energy_bins)

    model = mc.Model(
        geometry=geometry,
        tallies=[tally],
        source=source,
        total_particles=PARTICLES * BATCHES,
        seed=1,
        transport_secondary_photons=True,
        use_decay_photons=True,
    )

    t0 = time.time()
    dp = model.simulate_transport(compute=compute)[tally]
    elapsed = time.time() - t0

    # Build the irradiation/cooling schedule (Pulse for irrad, Cooldown for cool)
    # and time-correct the parent-nuclide-binned photon tally into a DoseResult.
    schedule_steps = []
    for dt, rate in zip(D1S_TIMESTEPS, D1S_SOURCE_RATES):
        if rate > 0:
            schedule_steps.append(mc.Pulse(source=source, rate=rate, duration=dt))
        else:
            schedule_steps.append(mc.Cooldown(duration=dt))
    schedule = mc.PulseSchedule(schedule_steps)
    dose = schedule.time_correct_tally(dp)

    spectra = [list(row) for row in np.array(dose.mean)]
    spectra_std = [list(row) for row in np.array(dose.std_dev)]

    return {
        'backend': f'yamc-{compute}',
        'radionuclides': radionuclides,
        'energy_bins': energy_bins,
        'spectra': spectra,
        'spectra_std': spectra_std,
        'elapsed': elapsed,
    }, model


In [5]:
def create_step_trace(energy_bins, values, name, color, dash=None):
    """Create a step plot trace for Plotly."""
    x_step = []
    y_step = []
    for i in range(len(values)):
        x_step.extend([energy_bins[i], energy_bins[i+1]])
        y_step.extend([values[i], values[i]])

    return go.Scatter(
        x=x_step,
        y=y_step,
        mode='lines',
        name=name,
        line=dict(color=color, width=2, dash=dash),
    )


def create_comparison_plot(energy_bins, openmc_flux, openmc_std, yamc_flux, yamc_std, title="Flux"):
    """Create a Plotly subplot with photon flux spectra (top) and ratio (bottom)."""

    energy_bins = np.asarray(energy_bins)
    bin_centers = np.sqrt(energy_bins[:-1] * energy_bins[1:])

    omc_flux = np.asarray(openmc_flux, dtype=float).copy()
    omc_std = np.asarray(openmc_std, dtype=float).copy()
    ymc_flux = np.asarray(yamc_flux, dtype=float).copy()
    ymc_std = np.asarray(yamc_std, dtype=float).copy()

    # Normalize by lethargy bin width
    lethargy_bin_width = np.log10(energy_bins[1:] / energy_bins[:-1])
    omc_flux /= lethargy_bin_width
    omc_std /= lethargy_bin_width
    ymc_flux /= lethargy_bin_width
    ymc_std /= lethargy_bin_width

    has_flux = (omc_flux > 0) | (ymc_flux > 0)
    if np.any(has_flux):
        first_nonzero_idx = np.argmax(has_flux)
        min_plot_energy = energy_bins[first_nonzero_idx]
    else:
        min_plot_energy = energy_bins[0]

    max_plot_energy = energy_bins[-1] * 1.1

    max_flux = max(
        np.max(omc_flux[omc_flux > 0]) if np.any(omc_flux > 0) else 1e-10,
        np.max(ymc_flux[ymc_flux > 0]) if np.any(ymc_flux > 0) else 1e-10
    )
    flux_threshold = max_flux / (10 ** FLUX_DECADES)

    omc_flux_plot = np.where(omc_flux >= flux_threshold, omc_flux, np.nan)
    ymc_flux_plot = np.where(ymc_flux >= flux_threshold, ymc_flux, np.nan)
    omc_std_plot = np.where(omc_flux >= flux_threshold, omc_std, np.nan)
    ymc_std_plot = np.where(ymc_flux >= flux_threshold, ymc_std, np.nan)

    valid = (omc_flux > 0) & (ymc_flux > 0)
    with np.errstate(divide='ignore', invalid='ignore'):
        ratio = np.where(valid, ymc_flux / omc_flux, np.nan)
        rel_err_yamc = np.where(ymc_flux > 0, ymc_std / ymc_flux, 0)
        rel_err_openmc = np.where(omc_flux > 0, omc_std / omc_flux, 0)
    ratio_rel_err = np.sqrt(rel_err_yamc**2 + rel_err_openmc**2)
    ratio_std = np.where(valid, ratio * ratio_rel_err, np.nan)

    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.08,
        row_heights=[0.7, 0.3],
        subplot_titles=('Photon Flux per Unit Lethargy', 'YAMC / OpenMC Ratio')
    )

    fig.add_trace(
        create_step_trace(energy_bins, omc_flux_plot, 'OpenMC', 'blue'),
        row=1, col=1
    )
    fig.add_trace(
        create_step_trace(energy_bins, ymc_flux_plot, 'YAMC', 'red', dash='dash'),
        row=1, col=1
    )

    fig.add_trace(
        go.Scatter(
            x=bin_centers, y=omc_flux_plot, mode='markers',
            marker=dict(color='blue', size=4, symbol='circle'),
            error_y=dict(type='data', array=omc_std_plot, visible=True, color='blue', thickness=1),
            showlegend=False,
        ),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(
            x=bin_centers, y=ymc_flux_plot, mode='markers',
            marker=dict(color='red', size=4, symbol='square'),
            error_y=dict(type='data', array=ymc_std_plot, visible=True, color='red', thickness=1),
            showlegend=False,
        ),
        row=1, col=1
    )

    fig.add_trace(
        create_step_trace(energy_bins, ratio, 'YAMC/OpenMC', 'green'),
        row=2, col=1
    )
    fig.add_hline(y=1.0, line_dash="dash", line_color="gray", row=2, col=1)

    fig.update_xaxes(type="log", title_text="Energy (eV)",
                     range=[np.log10(min_plot_energy), np.log10(max_plot_energy)],
                     exponentformat='e', row=2, col=1)
    fig.update_xaxes(type="log",
                     range=[np.log10(min_plot_energy), np.log10(max_plot_energy)],
                     exponentformat='e', row=1, col=1)
    fig.update_yaxes(type="log", title_text="Photon Flux / Lethargy", exponentformat='e', row=1, col=1)
    fig.update_yaxes(title_text="Ratio", range=[0.5, 1.5], row=2, col=1)

    fig.update_layout(
        height=700,
        title=title,
        legend=dict(x=0.02, y=0.98),
        margin=dict(l=60, r=20, t=60, b=40)
    )

    return fig


def calculate_spectral_agreement(openmc_flux, yamc_flux, openmc_std, yamc_std, flux_threshold=1e-6):
    """Calculate statistical agreement metrics for a single photon flux spectrum."""

    openmc_flux = np.asarray(openmc_flux, dtype=float)
    yamc_flux = np.asarray(yamc_flux, dtype=float)
    openmc_std = np.asarray(openmc_std, dtype=float)
    yamc_std = np.asarray(yamc_std, dtype=float)

    chi_squared_terms = []
    rel_diffs = []
    weighted_rel_diffs = []
    flux_weights = []

    for i in range(len(openmc_flux)):
        omc, ymc = openmc_flux[i], yamc_flux[i]
        omc_s, ymc_s = openmc_std[i], yamc_std[i]
        combined_std = np.sqrt(omc_s**2 + ymc_s**2)
        if combined_std > 0:
            chi_squared_terms.append(((ymc - omc) / combined_std) ** 2)
        if omc > flux_threshold or ymc > flux_threshold:
            if omc > 0:
                rd = abs((ymc - omc) / omc) * 100
                if np.isfinite(rd):
                    rel_diffs.append(rd)
                    w = max(omc, ymc)
                    weighted_rel_diffs.append(rd * w)
                    flux_weights.append(w)

    chi_squared = float(np.sum(chi_squared_terms))
    dof = len(chi_squared_terms)
    reduced_chi_squared = chi_squared / dof if dof > 0 else 0.0
    median_rel_diff = float(np.median(rel_diffs)) if rel_diffs else 0.0
    weighted_avg_rel_diff = (
        float(np.sum(weighted_rel_diffs) / np.sum(flux_weights)) if flux_weights else 0.0
    )

    if reduced_chi_squared < 1.5:
        quality = "Excellent"
    elif reduced_chi_squared < 3.0:
        quality = "Good"
    elif reduced_chi_squared < 5.0:
        quality = "Fair"
    else:
        quality = "Poor"

    return {
        'reduced_chi_squared': reduced_chi_squared,
        'degrees_of_freedom': dof,
        'median_rel_diff': median_rel_diff,
        'weighted_avg_rel_diff': weighted_avg_rel_diff,
        'quality': quality,
        'bins_analyzed': len(rel_diffs),
        'total_bins': len(openmc_flux),
    }


def create_agreement_table(stats):
    """Create a summary table of spectral agreement statistics."""
    rows = [
        {'Metric': 'Reduced Chi-Squared', 'Value': f"{stats['reduced_chi_squared']:.2f}",
         'Note': f"(ideal ~ 1.0, dof={stats['degrees_of_freedom']})"},
        {'Metric': 'Weighted Avg Rel. Diff', 'Value': f"{stats['weighted_avg_rel_diff']:.2f}%",
         'Note': '(flux-weighted)'},
        {'Metric': 'Median Rel. Diff', 'Value': f"{stats['median_rel_diff']:.2f}%",
         'Note': '(robust to outliers)'},
        {'Metric': 'Agreement Quality', 'Value': stats['quality'],
         'Note': f"({stats['bins_analyzed']}/{stats['total_bins']} bins analyzed)"},
    ]
    return pd.DataFrame(rows)


def d1s_quality_label(max_rel_diff):
    """Assign quality label based on max relative difference (%) for D1S."""
    if max_rel_diff < 2.0:
        return "Excellent"
    elif max_rel_diff < 5.0:
        return "Good"
    elif max_rel_diff < 10.0:
        return "Fair"
    else:
        return "Poor"


def rel_diff(a, b):
    """Safe relative difference: (b - a) / a * 100, returns NaN if a == 0."""
    if a == 0:
        return np.nan
    return (b - a) / a * 100


def compute_step_metrics(openmc_result, yamc_result):
    """Compute per-timestep comparison metrics."""
    step_metrics = []
    for step_idx in range(len(D1S_TIMESTEPS)):
        omc_spec = np.array(openmc_result['spectra'][step_idx])
        ymc_spec = np.array(yamc_result['spectra'][step_idx])

        omc_total = omc_spec.sum()
        ymc_total = ymc_spec.sum()
        total_rel = rel_diff(omc_total, ymc_total)

        step_metrics.append({
            'step': step_idx + 1,
            'phase': D1S_PHASES[step_idx],
            'omc_total': float(omc_total),
            'ymc_total': float(ymc_total),
            'total_rel_pct': float(total_rel),
        })

    finite_diffs = [
        abs(m['total_rel_pct']) for m in step_metrics
        if np.isfinite(m['total_rel_pct'])
    ]
    max_total_rel = max(finite_diffs) if finite_diffs else 0.0
    quality = d1s_quality_label(max_total_rel)
    speedup = openmc_result['elapsed'] / yamc_result['elapsed'] if yamc_result['elapsed'] > 0 else float('inf')

    return {
        'step_metrics': step_metrics,
        'max_total_rel_pct': max_total_rel,
        'quality': quality,
        'speedup': speedup,
    }


def create_step_metrics_table(metrics):
    """Create a table of per-timestep comparison metrics."""
    rows = []
    for m in metrics['step_metrics']:
        rows.append({
            'Step': m['step'],
            'Phase': m['phase'],
            'OpenMC Total': f"{m['omc_total']:.4e}",
            'YAMC Total': f"{m['ymc_total']:.4e}",
            'Rel. Diff (%)': f"{m['total_rel_pct']:+.2f}%" if np.isfinite(m['total_rel_pct']) else 'N/A',
        })

    rows.append({
        'Step': '',
        'Phase': '',
        'OpenMC Total': '',
        'YAMC Total': 'Quality:',
        'Rel. Diff (%)': metrics['quality'],
    })

    return pd.DataFrame(rows)

In [6]:
# Discover D1S-relevant nuclides: those that produce radionuclides with decay photon sources
def element_from_nuclide(nuclide):
    """Extract element symbol from nuclide name: 'Fe56' -> 'Fe'."""
    return re.sub(r"\d+", "", nuclide)


def find_d1s_nuclides():
    """Find nuclides that produce radionuclides with decay photon sources.

    Reads the OpenMC chain XML when present (the original reference path); when
    that file is unavailable, derives the same half-life / photon-source /
    reaction-target maps from the YAMC transmutation chain so the notebook can
    discover nuclides without the OpenMC data tree (e.g. MC_BACKENDS without
    openmc-cpu).
    """
    half_lives = {}
    reactions_map = {}
    photon_sources = set()

    if os.path.exists(CHAIN_FILE_OPENMC):
        tree = ET.parse(CHAIN_FILE_OPENMC)
        root = tree.getroot()
        for nuc_elem in root.findall('nuclide'):
            name = nuc_elem.get('name')
            hl = nuc_elem.get('half_life')
            if hl is not None:
                half_lives[name] = float(hl)
            for source in nuc_elem.findall('source'):
                if source.get('particle') == 'photon':
                    photon_sources.add(name)
            for rx in nuc_elem.findall('reaction'):
                target = rx.get('target')
                reactions_map.setdefault(name, []).append(target)
    else:
        chain = yamc.TransmutationChain(TRANSMUTATION_DATA)
        half_lives = dict(chain.half_lives)
        photon_sources = set(chain.photon_sources)
        # YAMC chain reactions are (kind, target, branching) tuples; the
        # OpenMC XML path stores a flat list of target names, so extract
        # the target name to keep the downstream matching identical.
        reactions_map = {
            name: [t[1] for t in entries]
            for name, entries in chain.reactions.items()
        }

    d1s_parents = set()
    for parent, targets in reactions_map.items():
        for target in targets:
            if target and target != 'None':
                if target in half_lives and target in photon_sources:
                    d1s_parents.add(parent)

    neutron_dir = os.path.join(NUCLEAR_DATA_DIR[NUCLEAR_DATA], 'neutron')
    photon_dir = os.path.join(NUCLEAR_DATA_DIR[NUCLEAR_DATA], 'photon')
    h5_files = sorted(glob.glob(os.path.join(neutron_dir, '*.h5')))

    result = []
    for f in h5_files:
        name = os.path.splitext(os.path.basename(f))[0]
        if '_m' in name:
            continue
        if name not in d1s_parents:
            continue
        element = element_from_nuclide(name)
        photon_path = os.path.join(photon_dir, f"{element}.h5")
        if not os.path.exists(photon_path):
            continue
        result.append((name, f, element, photon_path))

    return result


nuclides = find_d1s_nuclides()

if not nuclides:
    raise FileNotFoundError(
        f"No D1S-relevant nuclides found.\n"
        f"Please check that the chain file and nuclear data directories exist."
    )

# Select which nuclide(s) to run. Honors (priority): the explicit
# YAMC_VERIFY_NUCLIDES list, then the YAMC_VERIFY_NUMBER_OF_SIMULATIONS
# count cap, else the full discovered list.
from cache_utils import select_requested
nuclides = select_requested(nuclides, kind="nuclide", key=lambda t: t[0])

print(f"Running D1S simulations for {len(nuclides)} nuclides...")
for nuc, _, elem, _ in nuclides:
    print(f"  {nuc} ({elem})")

Running D1S simulations for 1 nuclides...
  Fe56 (Fe)


In [7]:
# Run all simulations and collect results. OpenMC (cached) and each YAMC compute
# run in their own try/except -- openmc first, so it caches even when no yamc
# backend is selected, and a failing yamc nuclide never blocks the OpenMC cache.
all_results = {}
first_yamc_model = None

for nuclide, neutron_path, element, photon_path in nuclides:
    print(f"\nProcessing {nuclide} ({element})...")

    openmc_result = None
    if RUN_OPENMC:
        try:
            print("  Running OpenMC D1S...")
            openmc_result = cached_or_run(
                CACHE_DIR, NUCLEAR_DATA, "openmc", nuclide,
                lambda n=nuclide, npath=neutron_path, el=element, ppath=photon_path:
                    run_openmc_d1s(n, npath, el, ppath),
                meta=OPENMC_CACHE_META,
            )
            print(f"    {openmc_result['elapsed']:.1f}s, "
                  f"{len(openmc_result['radionuclides'])} radionuclides")
        except Exception as e:
            print(f"  OpenMC Error: {e}")

    yamc_by_compute = {}
    for compute in COMPUTES:
        try:
            print(f"  Running YAMC D1S ({compute})...")
            yamc_result, yamc_model = run_yamc_d1s(
                nuclide, neutron_path, element, photon_path, compute=compute,
            )
            yamc_by_compute[compute] = yamc_result
            if first_yamc_model is None:
                first_yamc_model = yamc_model
            print(f"    {yamc_result['elapsed']:.1f}s, "
                  f"{len(yamc_result['radionuclides'])} radionuclides")
        except Exception as e:
            print(f"  YAMC ({compute}) Error: {e}")

    # Nothing ran for this nuclide -> skip.
    if openmc_result is None and not yamc_by_compute:
        continue

    # Reference for the metrics: OpenMC if available, else yamc-cpu.
    reference = openmc_result if openmc_result is not None else yamc_by_compute.get("cpu")
    ref_label = "OpenMC" if openmc_result is not None else "yamc-cpu"

    per_compute_metrics = {}
    for compute, yamc_result in yamc_by_compute.items():
        if reference is None or (openmc_result is None and compute == "cpu"):
            # No reference (cpu-only, openmc off) -> nothing to compare against.
            continue
        metrics = compute_step_metrics(reference, yamc_result)
        per_compute_metrics[compute] = metrics
        print(f"  yamc-{compute} vs {ref_label}:")
        for m in metrics['step_metrics']:
            print(f"    Step {m['step']} ({m['phase']:>5}): "
                  f"{ref_label}={m['omc_total']:.4e}  yamc-{compute}={m['ymc_total']:.4e}  "
                  f"diff={m['total_rel_pct']:+.2f}%")
        print(f"    Quality: {metrics['quality']} | Speedup: {metrics['speedup']:.2f}x")

    all_results[nuclide] = {
        'openmc': openmc_result,
        'yamc': yamc_by_compute,
        'metrics': per_compute_metrics,
        'reference_label': ref_label,
        'element': element,
    }

print(f"\nCompleted {len(all_results)} / {len(nuclides)} nuclides successfully.")


Processing Fe56 (Fe)...
  Running OpenMC D1S...


                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######################     %%%%%%%%%%%%%%%%%%%%
                #######################     %%%%%%%%%%%%%%%%%%
                 #######################     %%%%%%%%%%%%%%%%%
                 #####################

 Reading Fe from
 /home/jon/nuclear_data/endf-b8.1-hdf5/endfb-viii.1-hdf5/photon/Fe.h5 
 Minimum neutron data temperature: 294.0 K
 Maximum neutron data temperature: 294.0 K
 Preparing distributed cell instances...
 Maximum neutron transport energy: 150000000.0 eV for Fe56

 ===============>     FIXED SOURCE TRANSPORT SIMULATION     <===============

 Simulating batch 1


 Simulating batch 2


 Simulating batch 3


 Simulating batch 4


... (10 outputs left out)

Progress: 35000000/50000000 particles (70%)  elapsed 747 ms, ETA ~320 ms

Progress: 50000000/50000000 particles (100%)  elapsed 1.0s, ETA ~0 ms   


    1.2s, 3 radionuclides
  Running YAMC D1S (gpu)...
Transport simulation complete.
Time: 1.215 s (data load: 0.011 s, transport: 1.202 s) | Particles/s: 41,168,341 (transport only: 41,588,281) | Total: 50,000,000 particles
Tally statistics:
  d1s_spectrum: FOM = 5.451e5 (per-bin: min=1.03e0, median=2.18e3, max=3.60e5)  checks=PASSED (7/7)
GPU (coupled n->photon): Vulkan "AMD Radeon Graphics (RADV STRIX_HALO)" (IntegratedGpu, vendor 0x1002)


    58.6s, 3 radionuclides
  yamc-cpu vs OpenMC:
    Step 1 (irrad): OpenMC=8.6949e-04  yamc-cpu=8.6854e-04  diff=-0.11%
    Step 2 (irrad): OpenMC=1.5341e-03  yamc-cpu=1.5324e-03  diff=-0.11%
    Step 3 (irrad): OpenMC=2.0420e-03  yamc-cpu=2.0397e-03  diff=-0.11%
    Step 4 ( cool): OpenMC=1.9225e-09  yamc-cpu=1.9243e-09  diff=+0.09%
    Step 5 ( cool): OpenMC=1.9027e-09  yamc-cpu=1.9044e-09  diff=+0.09%
    Quality: Excellent | Speedup: 4.37x
  yamc-gpu vs OpenMC:
    Step 1 (irrad): OpenMC=8.6949e-04  yamc-gpu=8.7125e-04  diff=+0.20%
    Step 2 (irrad): OpenMC=1.5341e-03  yamc-gpu=1.5372e-03  diff=+0.20%
    Step 3 (irrad): OpenMC=2.0420e-03  yamc-gpu=2.0461e-03  diff=+0.20%
    Step 4 ( cool): OpenMC=1.9225e-09  yamc-gpu=1.9220e-09  diff=-0.03%
    Step 5 ( cool): OpenMC=1.9027e-09  yamc-gpu=1.9021e-09  diff=-0.03%
    Quality: Excellent | Speedup: 0.09x

Completed 1 / 1 nuclides successfully.


In [8]:
# Display broomstick model geometry with source from first simulation
import base64
from IPython.display import HTML

if first_yamc_model is not None:
    html_plot = first_yamc_model.plot(basis="xz", n_samples=500, axis_units="cm")
    html_plot.save("d1s_model_plot.html")
    encoded = base64.b64encode(html_plot.html.encode()).decode()
    display(HTML(f'<iframe src="data:text/html;base64,{encoded}" width="100%" height="600" frameborder="0"></iframe>'))
else:
    print("No successful YAMC simulation to plot.")

/home/jon/yamc-org/yamc-verification/.venv/lib/python3.13/site-packages/IPython/core/display.py:447: UserWarning: Consider using IPython.display.IFrame instead
  warnings.warn("Consider using IPython.display.IFrame instead")


## Results by Nuclide

Below are the comparison results for each nuclide. Each section includes:

- TCF-corrected photon flux spectrum plot per timestep (with lethargy normalization)
- Per-timestep comparison metrics table
- Quality assessment
- Performance comparison

In [9]:
# Display results for each nuclide.
#
# `results['yamc']` is a dict keyed by compute ('cpu' / 'gpu'); `results['metrics']`
# holds per-compute step metrics against the reference (OpenMC if run, else
# yamc-cpu). Each YAMC backend is plotted against that reference.
for nuclide, results in all_results.items():
    openmc_result = results['openmc']
    yamc_by_compute = results['yamc']
    per_compute_metrics = results['metrics']
    ref_label = results['reference_label']
    element = results['element']

    # Reference spectra: OpenMC if present, else yamc-cpu.
    reference = openmc_result if openmc_result is not None else yamc_by_compute.get('cpu')
    if reference is None:
        continue

    display(Markdown(f"## {nuclide} ({element})"))

    ref_time = reference['elapsed']
    perf_bits = [f"{ref_label}: {ref_time:.1f}s"]
    for compute, yr in yamc_by_compute.items():
        perf_bits.append(f"yamc-{compute}: {yr['elapsed']:.1f}s")
    display(Markdown(
        "**Performance:** " + " | ".join(perf_bits) + "  \n"
        f"**Radionuclides:** {ref_label}: {len(reference['radionuclides'])}"
    ))

    energy_bins = reference['energy_bins']

    for compute, yamc_result in yamc_by_compute.items():
        metrics = per_compute_metrics.get(compute)
        if metrics is None:
            # yamc-cpu IS the reference (openmc off) -> nothing to compare.
            continue
        display(Markdown(f"### yamc-{compute} vs {ref_label}"))
        metrics_table = create_step_metrics_table(metrics)
        display(metrics_table.style.hide(axis='index'))

        for step_idx in range(len(D1S_TIMESTEPS)):
            phase = D1S_PHASES[step_idx]
            display(Markdown(f"#### Step {step_idx + 1} ({phase})"))

            omc_flux = np.array(reference['spectra'][step_idx])
            omc_std = np.array(reference['spectra_std'][step_idx])
            ymc_flux = np.array(yamc_result['spectra'][step_idx])
            ymc_std = np.array(yamc_result['spectra_std'][step_idx])

            fig = create_comparison_plot(
                energy_bins, omc_flux, omc_std, ymc_flux, ymc_std,
                title=f"{nuclide} - yamc-{compute} - Step {step_idx + 1} ({phase})",
            )
            fig.show()

            display(Markdown("##### Spectral Agreement"))
            stats = calculate_spectral_agreement(omc_flux, ymc_flux, omc_std, ymc_std)
            display(create_agreement_table(stats).style.hide(axis='index'))

    print()


## Fe56 (Fe)

**Performance:** OpenMC: 5.3s | yamc-cpu: 1.2s | yamc-gpu: 58.6s  
**Radionuclides:** OpenMC: 3

### yamc-cpu vs OpenMC

Step,Phase,OpenMC Total,YAMC Total,Rel. Diff (%)
1,irrad,8.6949e-04,8.6854e-04,-0.11%
2,irrad,1.5341e-03,1.5324e-03,-0.11%
3,irrad,2.0420e-03,2.0397e-03,-0.11%
4,cool,1.9225e-09,1.9243e-09,+0.09%
5,cool,1.9027e-09,1.9044e-09,+0.09%
,,,Quality:,Excellent


#### Step 1 (irrad)

##### Spectral Agreement

... (26 outputs left out)

Metric,Value,Note
Reduced Chi-Squared,129.92,"(ideal ~ 1.0, dof=27)"
Weighted Avg Rel. Diff,0.00%,(flux-weighted)
Median Rel. Diff,0.00%,(robust to outliers)
Agreement Quality,Poor,(0/42 bins analyzed)


#### Step 5 (cool)

##### Spectral Agreement

Metric,Value,Note
Reduced Chi-Squared,129.92,"(ideal ~ 1.0, dof=27)"
Weighted Avg Rel. Diff,0.00%,(flux-weighted)
Median Rel. Diff,0.00%,(robust to outliers)
Agreement Quality,Poor,(0/42 bins analyzed)
